Structuur nieuwe voorspellingen maken:
1. Verkrijg de kijkcijfer (KC) data van laatste 2-3 weken (EXCLUSIEF DE GEWESTE PREDICTIES)
2. Clean KC data
3. Haal de weerdata op van minDate tot maxDate van de KC data
4. Merge weerdata en KC data
5. Drop missing values
6. Feature engineering
   1. Timestamp
   2. Lag features (haal deze uit de opgehaalde data)
   3. Target Encoding (haal deze uit opgehaalde data)
7. Gebruik model om te predicten ADHV de data

### Fetch Kijkcijferdata

In [11]:
def fetch_kijkcijfers(start_date, end_date):
    data_list = []
    print(f"Start fetching kijkcijfers van {start_date.date()} tot {end_date.date()}")

    # Loop door elke dag
    current_date = start_date
    while current_date <= end_date:
        datum = f"{current_date.year}-{current_date.month}-{current_date.day}"
        url = f"https://api.cim.be/api/cim_tv_public_results_daily_views?dateDiff={datum}&reportType=north"
        
        try:
            response = requests.get(url)
            if response.status_code == 200:
                data = response.json()
                programma_lijst = data.get('hydra:member', [])
                
                for programma in programma_lijst:
                    try:
                        data_list.append({
                            'dateDiff': programma.get('dateDiff'),
                            'ranking': programma.get('ranking'),
                            'description': programma.get('description'),
                            'channel': programma.get('channel'),
                            'startTime': programma.get('startTime'),
                            'rLength': programma.get('rLength'),
                            'rateInK': programma.get('rateInK'),
                            'live': programma.get('live')
                        })
                        
                    except Exception as e:
                        print(f"Fout bij verwerken programma op {datum}: {e}")
            else:
                print(f"Geen data voor {datum} (HTTP {response.status_code})")
                
        except Exception as e:
            print(f"Fout bij ophalen {datum}: {e}")
        
        current_date += timedelta(days=1)
    
    print(f"Einde fetching kijkcijfers")
    # Maak een dataframe van de data
    df = pd.DataFrame(data_list)
    return df

# Voorbeeld van hoe de functie te gebruiken
end_date = datetime.today()
start_date = end_date - timedelta(weeks=3)

df_kijkcijfers = fetch_kijkcijfers(start_date, end_date)
df_kijkcijfers

Start fetching kijkcijfers van 2025-03-04 tot 2025-03-25
Einde fetching kijkcijfers


,dateDiff,ranking,description,channel,startTime,rLength,rateInK,live
0,2025-03-04T00:00:00.000000,1,THUIS,VRT 1,20:17:58,00:24:12,975.426,7
1,2025-03-04T00:00:00.000000,2,FACTCHECKERS,VRT 1,20:44:45,00:47:24,900.885,7
2,2025-03-04T00:00:00.000000,3,HET 7 UUR-JOURNAAL,VRT 1,19:00:04,00:45:56,844.846,7
3,2025-03-04T00:00:00.000000,4,MAN BIJT HOND,VRT 1,19:49:05,00:22:00,567.600,7
4,2025-03-04T00:00:00.000000,5,FAMILIE,VTM,20:09:18,00:24:46,557.124,7
...,...,...,...,...,...,...,...,...
415,2025-03-24T00:00:00.000000,16,DE AFSPRAAK,VRT CANVAS,20:37:48,00:46:10,209.299,0
416,2025-03-24T00:00:00.000000,17,DE RIDDER,VRT 1,17:10:49,00:49:51,207.146,0
417,2025-03-24T00:00:00.000000,18,MILO,VTM,18:24:47,00:22:54,189.752,0
418,2025-03-24T00:00:00.000000,19,ONDERZOEKSRECHTERS,VTM,21:55:35,00:46:33,174.848,0


### Fetch weerdata

In [12]:
def fetch_weerdata(start_date, end_date):
    # Locatie voor Vlaanderen (Brussel als centraal punt)
    latitude = 50.8503
    longitude = 4.3517

    # API endpoints voor historische data en forecast
    archive_url = "https://archive-api.open-meteo.com/v1/archive"
    forecast_url = "https://api.open-meteo.com/v1/forecast"

    # Relevante hourly variabelen voor het ML model
    hourly_vars = [
        "temperature_2m",   # Gemiddelde temperatuur per uur
        "weathercode",      # Weertype als code per uur
        "precipitation",    # Totale neerslag per uur
        "rain",             # Regen per uur
        "snowfall",         # Sneeuwval per uur
        "cloudcover",       # Bewolking per uur
        "windspeed_10m"     # Windsnelheid per uur
    ]
    
    # Gemeenschappelijke API parameters
    common_params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": hourly_vars,
        "timezone": "Europe/Brussels",
        "temperature_unit": "celsius",
        "precipitation_unit": "mm",
        "windspeed_unit": "kmh"
    }
    
    today = datetime.today().date()
    dataframes = []
    
    # Historische data (als start_date < vandaag)
    if start_date.date() < today:
        # Bepaal de einddatum voor historische data (niet later dan gisteren)
        hist_end_date = min(end_date.date(), today - timedelta(days=1))
        params_hist = common_params.copy()
        params_hist.update({
            "start_date": start_date.strftime('%Y-%m-%d'),
            "end_date": hist_end_date.strftime('%Y-%m-%d')
        })
        print(f"Fetching historische weerdata van {params_hist['start_date']} tot {params_hist['end_date']}")
        response = requests.get(archive_url, params=params_hist)
        if response.status_code == 200:
            data = response.json().get("hourly", {})
            df_hist = pd.DataFrame({
                "time": data.get("time", []),
                "temperature": data.get("temperature_2m", []),
                "weather_code": data.get("weathercode", []),
                "precipitation": data.get("precipitation", []),
                "rain": data.get("rain", []),
                "snowfall": data.get("snowfall", []),
                "cloudcover": data.get("cloudcover", []),
                "windspeed": data.get("windspeed_10m", [])
            })
            if not df_hist.empty:
                df_hist['time'] = pd.to_datetime(df_hist['time'])
                df_hist['hour'] = df_hist['time'].dt.hour              # Uur van de dag
                df_hist['day_of_week'] = df_hist['time'].dt.dayofweek      # 0 = maandag, 6 = zondag
                df_hist['month'] = df_hist['time'].dt.month                # Maand (1 t/m 12)
                df_hist['year'] = df_hist['time'].dt.year                  # Jaar
                dataframes.append(df_hist)
        else:
            print(f"Fout bij het ophalen van historische data: {response.status_code}")
            print(response.text)
    
    # Forecast data (als end_date >= vandaag)
    if end_date.date() >= today:
        # Voor forecast gebruiken we data vanaf vandaag (of start_date als deze later is dan vandaag)
        forecast_start = max(start_date, datetime.combine(today, datetime.min.time()))
        params_forecast = common_params.copy()
        params_forecast.update({
            "start_date": forecast_start.strftime('%Y-%m-%d'),
            "end_date": end_date.strftime('%Y-%m-%d')
        })
        print(f"Fetching forecast weerdata van {params_forecast['start_date']} tot {params_forecast['end_date']}")
        response = requests.get(forecast_url, params=params_forecast)
        if response.status_code == 200:
            data = response.json().get("hourly", {})
            df_forecast = pd.DataFrame({
                "time": data.get("time", []),
                "temperature": data.get("temperature_2m", []),
                "weather_code": data.get("weathercode", []),
                "precipitation": data.get("precipitation", []),
                "rain": data.get("rain", []),
                "snowfall": data.get("snowfall", []),
                "cloudcover": data.get("cloudcover", []),
                "windspeed": data.get("windspeed_10m", [])
            })
            if not df_forecast.empty:
                df_forecast['time'] = pd.to_datetime(df_forecast['time'])
                df_forecast['hour'] = df_forecast['time'].dt.hour
                df_forecast['day_of_week'] = df_forecast['time'].dt.dayofweek
                df_forecast['month'] = df_forecast['time'].dt.month
                df_forecast['year'] = df_forecast['time'].dt.year
                dataframes.append(df_forecast)
        else:
            print(f"Fout bij het ophalen van forecast data: {response.status_code}")
            print(response.text)
    
    if dataframes:
        # Combineer en sorteer de dataframes op tijd
        df = pd.concat(dataframes).sort_values("time").reset_index(drop=True)
    else:
        df = pd.DataFrame()
    
    print("Einde fetching weerdata")
    return df

# Voorbeeld van hoe de functie te gebruiken:
start_date = datetime.today() - timedelta(weeks=3)
end_date = datetime.today() + timedelta(days=2)

fetch_weerdata(start_date, end_date)

Fetching historische weerdata van 2025-03-04 tot 2025-03-24
Fetching forecast weerdata van 2025-03-25 tot 2025-03-27
Einde fetching weerdata


,time,temperature,weather_code,precipitation,rain,snowfall,cloudcover,windspeed,hour,day_of_week,month,year
0,2025-03-04 00:00:00,2.6,3.0,0.0,0.0,0.0,99.0,4.8,0,1,3,2025
1,2025-03-04 01:00:00,2.0,3.0,0.0,0.0,0.0,100.0,4.0,1,1,3,2025
2,2025-03-04 02:00:00,2.2,3.0,0.0,0.0,0.0,100.0,4.7,2,1,3,2025
3,2025-03-04 03:00:00,1.6,3.0,0.0,0.0,0.0,99.0,4.8,3,1,3,2025
4,2025-03-04 04:00:00,1.1,3.0,0.0,0.0,0.0,100.0,4.8,4,1,3,2025
...,...,...,...,...,...,...,...,...,...,...,...,...
571,2025-03-27 19:00:00,16.2,0.0,0.0,0.0,0.0,0.0,7.6,19,3,3,2025
572,2025-03-27 20:00:00,15.0,0.0,0.0,0.0,0.0,0.0,3.2,20,3,3,2025
573,2025-03-27 21:00:00,14.1,0.0,0.0,0.0,0.0,0.0,2.9,21,3,3,2025
574,2025-03-27 22:00:00,13.1,0.0,0.0,0.0,0.0,0.0,4.3,22,3,3,2025
